# TASK 5 & 6 - Değerlendirme ve Konuşma Geçmişi (Multi-Turn) Senaryoları
Bu notebook'ta:
1. Ragas kullanılarak pipeline performansı ölçülmektedir.
2. Konuşma Geçmişi bazlı multi-turn senaryolar simüle edilmektedir.

In [5]:
import sys
import os
import json
import warnings
import pandas as pd
from datasets import Dataset

warnings.filterwarnings("ignore")

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

from rag_pipeline import ask, get_collection, conv_manager
from eval_llm_wrapper import RotatingGeminiLLM
from eval_embed_wrapper import HFEmbeddings

from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy
)


## 1. Test Setini Yükleme ve RAG Cevapları

In [3]:
test_set_path = os.path.join("..", "evaluation", "test_set.json")
with open(test_set_path, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

collection = get_collection()

questions = []
ground_truths = []
answers = []
contexts = []

import time

for item in eval_data:
    q = item["question"]
    gt = item["ground_truth"]
    
    result = ask(q, collection, session_id="eval_session")
    
    questions.append(q)
    ground_truths.append(gt)
    answers.append(result["answer"])
    contexts.append(result.get("contexts", []))
    
    # Bağımsız sorular arasında context karışmasını önlüyoruz
    conv_manager.clear_session("eval_session")
    
    # Gemini API Quota (15 RPM) değerini aşmamak için her istekten sonra 4 saniye bekle
    time.sleep(4)

print("Tüm cevaplar üretildi.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4176.32it/s]
ERROR:root:API Error with key (ending in ...vbtk): You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in 46.431772946s.. Retrying in 2.0s...
ERROR:root:API Error with key (ending in ...oPS0): You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in

KeyboardInterrupt: 

## 2. Ragas Metrikleri

In [ ]:
from ragas.run_config import RunConfig

data = {
    "user_input": questions,
    "response": answers,
    "retrieved_contexts": contexts,
    "reference": ground_truths
}

dataset = Dataset.from_dict(data)

eval_llm = RotatingGeminiLLM()
eval_embeddings = HFEmbeddings()

metrics = [
    Faithfulness(),
    AnswerRelevancy()
]

# Configure Ragas to run strictly sequentially + wait mechanisms
run_config = RunConfig(
    max_workers=1,
    max_retries=15, 
    max_wait=60
)

print("Starting evaluation...")
result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=eval_llm,
    embeddings=eval_embeddings,
    run_config=run_config
)

eval_df = result.to_pandas()
display(eval_df)

out_csv = os.path.join("..", "evaluation", "eval_results.csv")
eval_df.to_csv(out_csv, index=False)
print(f"Rapor CSV formatında kaydedildi: {out_csv}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4528.23it/s]


Starting evaluation...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

API Error with key ...vbtk: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 58.523339913s.. Retrying in 2.0s...
API Error with key ...oPS0: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 56.116364511s.. Retrying in 2.0s...
API Error with key ...UQ-s: You exceeded your current quota, please check your plan and 

ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
Evaluating:   2%|▎         | 1/40 [03:00<1:57:01, 180.03s/it]

API Error with key ...UQ-s: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 58.639121826s.. Retrying in 2.0s...
API Error with key ...vbtk: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 56.384380233s.. Retrying in 2.0s...
API Error with key ...oPS0: You exceeded your current quota, please check your plan and 

## 3. Multi-Turn Conversation (Konuşma Geçmişi) Senaryoları

In [6]:
# Eğer üstteki RAG test hücreleri çalıştırılmadıysa diye collection'ı burada da tanımlıyoruz
collection = get_collection()

def run_scenario(session_id, user_msgs):
    print(f"\n{'='*50}\nSenaryo Başlıyor: {session_id}\n{'='*50}")
    conv_manager.clear_session(session_id)
    
    for msg in user_msgs:
        print(f"\nKullanıcı: {msg}")
        result = ask(msg, collection, session_id=session_id)
        print(f"Asistan: {result['answer']}\n")
        
# Senaryo 1: Bağlamsal zamir kullanımı (TR)
run_scenario("scenario_1", [
    "Garanti süresi ne kadar?",
    "Küçük ev aletlerinde bu durumu nasıl uzatabilirim?"
])

# Senaryo 2: Bağlamsal zamir kullanımı (EN)
run_scenario("scenario_2", [
    "How much does standard shipping cost?",
    "What if my order is over $60?"
])

# Senaryo 3: Ürün değişimi sorgusu ve anaforik takip
run_scenario("scenario_3", [
    "Aldığım tişörtün bedeni uymadı, iade mi edeyim?",
    "Peki bu süreç nasıl işliyor ve ücreti var mı?",
    "Teşekkürler, çok yardımcı oldunuz."
])


Senaryo Başlıyor: scenario_1

Kullanıcı: Garanti süresi ne kadar?


ERROR:root:API Error with key (ending in ...UQ-s): You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in 11.461125856s.. Retrying in 2.0s...
ERROR:root:API Error with key (ending in ...vbtk): You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in 9.0486486s.. Retrying in 2.0s...
ERROR:root:API Error with key (endi

KeyboardInterrupt: 